# Python で海洋モデルデータを扱う：Xarray、Dask、Zarr、HEALPix

**概要**

魚の位置推定に入る前に、`pangeo-fish` がタグのデータと比べる「参照」データ、つまり 3 次元の海洋モデルに慣れておきましょう。
このノートブックでは、全球海洋再解析 **GLORYS**（Global Ocean Physics Reanalysis、1/12°、日平均）を [Copernicus Marine Service](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030) から直接開きます。そして、[Pangeo](https://pangeo.io) エコシステムの主要なツールを使って、データを眺め、図を描き、計算し、保存します。

- **Xarray**：ラベル付きの N 次元配列です（`data.frame` に似ていますが、時間・深さ・緯度・経度のような複数の次元を持てます）。
- **Dask**：計算を並列に、しかも必要になったときにだけ実行します（*遅延*評価）。そのため、メモリより大きなデータも扱えます。
- **Zarr**：N 次元配列のための、チャンクに分割されたクラウド向けのファイル形式です。Copernicus Marine はデータを Zarr で配信しており、`pangeo-fish` も途中結果を Zarr で保存します。
- **HEALPix**（離散全球格子システム、DGGS の一つ）：球面を等面積に分割する格子です。`pangeo-fish` はすべての計算をこの格子の上で行います。ここでは GLORYS を HEALPix に変換し、`xdggs` で描画します。

**所要時間**：約 25 分です。`Shift+Enter` でセルを 1 つずつ実行してください。引数をすべて理解する必要はありません。pangeo-fish のノートブックで再び出てくるオブジェクトに見覚えがある、という状態になることが目標です。

**カーネル**：**Python (pangeo-fish)** カーネルを使ってください（ノートブックの右上）。

## 0. R ユーザーの方へ：対応表

参加者の多くは R をお使いだと思います。R で知っていることと、今日 Python で目にするものの大まかな対応を示します。

| R では | Python では（今日） | 補足 |
|---|---|---|
| `library(dplyr)` | `import xarray as xr` | `xr` はよく使われる短縮名です |
| `data.frame` / `tibble` | `pandas.DataFrame` | 2 次元の表（今日はあまり使いません） |
| `dimnames` 付きの `array`、`raster`/`terra` の `SpatRaster`、`stars` オブジェクト | `xarray.DataArray` / `xarray.Dataset` | ラベル付きの N 次元配列です。`Dataset` は、座標を共有する複数の `DataArray` をまとめた辞書のようなものです |
| `filter(df, lat > 46)` または `df[df$lat > 46, ]` | `ds.sel(latitude=slice(46, 59))` | *ラベル*（座標の値）で選択します |
| `df[1:10, ]` | `ds.isel(time=slice(0, 10))` | *整数の位置*で選択します |
| `mean(x, na.rm = TRUE)` | `ds.mean("time")` | 集計は*名前の付いた*次元に沿って行います。NaN は既定で無視されます |
| `ggplot2` | `matplotlib`（`.plot()`）または `hvplot`（インタラクティブ） | |
| `future` / `parallel` | `dask` | 並列・遅延計算 |
| `ncdf4`、`terra::rast()` | `xr.open_dataset()`、`xr.open_zarr()` | ファイルの読み込み |
| `x <- 1` | `x = 1` | 代入 |
| `x$name` | `x["name"]` または `x.name` | 属性・要素へのアクセス |
| `TRUE`、`FALSE`、`NULL`、`NA` | `True`、`False`、`None`、`np.nan` | |

R ユーザーが戸惑いやすい点が 2 つあります。

1. Python では**インデント（字下げ）に意味があります**（`{ }` の代わりになります）。
2. **インデックスは 0 から始まります**。`slice(0, 10)` は位置 0 から 9 までを表します。

## 1. 準備

### ライブラリを読み込む

In [ ]:
import pathlib

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import hvplot.xarray  # xarray のオブジェクトに .hvplot アクセサーを追加する

import copernicusmarine
import xdggs
from healpix_resample import BilinearResampler

xr.set_options(display_expand_attrs=False)
print("xarray", xr.__version__, "| copernicusmarine", copernicusmarine.__version__, "| xdggs", xdggs.__version__)

### Dask クラスターを起動する

Dask は処理を多数の小さなタスクに分け、*ワーカー*の上で並列に実行します。Pangeo@EOSC のサーバーには複数の CPU があるので、それらを使う*ローカル*クラスターを起動します。

セルを実行したら、**Dashboard** のリンクをクリックしてください（または、JupyterLab の左サイドバーにある Dask タブを開き、`Task Stream` と `Progress` をクリックします）。このあとノートブックの中で計算が進む様子を*目で見る*ことができます。

In [ ]:
from dask.distributed import Client

client = Client()  # このマシンの CPU を使うローカルクラスター
client

### Copernicus Marine にログインする（初回のみ）

Copernicus Marine Service は無料ですが、アカウントが必要です（[登録はこちら](https://data.marine.copernicus.eu/register)）。
下のセルは、**初回だけ**ユーザー名とパスワードを尋ね、ホームディレクトリの `~/.copernicusmarine/.copernicusmarine-credentials` に保存します（Pangeo@EOSC のホームディレクトリは永続的です）。次にノートブックを開いたときは、何も聞かれません。

In [ ]:
credentials_file = pathlib.Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"

if credentials_file.exists():
    print("Copernicus Marine credentials found:", credentials_file)
else:
    copernicusmarine.login()  # ユーザー名とパスワードの入力を求められる

## 2. Xarray で GLORYS を開く

日平均の GLORYS 再解析（`dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m"`。`pangeo-fish` が既定の参照モデルとして使うものと同じデータセットです）を、次の条件で開きます。

- pangeo-fish のチュートリアルで使う**ケルト海・ビスケー湾・イギリス海峡**の範囲（`latitude` 北緯 46〜59 度、`longitude` 西経 14 度〜東経 2 度）
- **1 週間**分のデータ
- 変数 **`thetao`**（海水のポテンシャル水温）と **`zos`**（海面高度）

この時点では、まだ何もダウンロードされません。`copernicusmarine.open_dataset` は*メタデータ*だけを読み、遅延評価の `xarray.Dataset` を返します。

In [ ]:
%%time
bbox = {"latitude": [46, 59], "longitude": [-14, 2]}
time_slice = slice("2020-06-01", "2020-06-07")

glorys = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "zos"],
    minimum_longitude=bbox["longitude"][0],
    maximum_longitude=bbox["longitude"][1],
    minimum_latitude=bbox["latitude"][0],
    maximum_latitude=bbox["latitude"][1],
    start_datetime=time_slice.start,
    end_datetime=time_slice.stop,
)
glorys

少し時間をとって、上の HTML 表示を見てみましょう（三角のマークや右側のアイコンをクリックしてください）。

- **Dimensions（次元）**：`time`、`depth`、`latitude`、`longitude` とそれぞれの大きさ。
- **Coordinates（座標）**：各次元に沿ったラベル（日付、メートル単位の深さ、度）。
- **Data variables（データ変数）**：`thetao` は 4 次元（time、depth、latitude、longitude）、`zos` は 3 次元です。
- **Attributes（属性）**：メタデータ（単位、正式名称、出典など）。
- データ本体は **Dask 配列**です。*チャンク*の大きさとチャンクの数を見てください。各チャンクは、Copernicus のクラウド上にある Zarr ストアの、圧縮された小さな断片です。

<div class="alert alert-success">
<b>ポイント</b>
<ul>
<li><code>xarray.Dataset</code> は、次元と座標を共有する複数の変数をまとめて持ちます。</li>
<li>データセットを開くのは一瞬です。データは必要になったときに初めて読み込まれます。</li>
</ul>
</div>

### データの一部を選ぶ

`.sel()` では*座標の値*（ラベル）で、`.isel()` では*位置*で選択します。初日の海面水温（最も浅い深さの層、約 0.5 m）を取り出してみましょう。

In [ ]:
sst = glorys["thetao"].isel(depth=0)  # DataArray：time、latitude、longitude
sst_day1 = sst.sel(time="2020-06-01")
sst_day1

### 図を描く

`xarray` には `matplotlib` を使った `.plot()` メソッドがあり、次元名と単位が自動的に使われます。この 1 枚の地図のデータが Copernicus のクラウドからダウンロードされるのは、**この時点**です（Dask のダッシュボードを見てください）。

In [ ]:
sst_day1.plot(cmap="RdYlBu_r", figsize=(8, 6));

インタラクティブな地図（拡大・縮小、マウスを重ねて値を読む、時間のスライダー）には `hvplot` を使います。

In [ ]:
sst.hvplot(
    x="longitude", y="latitude", cmap="RdYlBu_r", geo=True, coastline=True,
    frame_width=500, title="GLORYS sea surface temperature (degC)",
)

### 1 地点の時系列

標識された魚は、水温と深度を記録します。この記録をモデルと比べるために、`pangeo-fish` は魚がいる可能性のある場所の近くのモデル値を調べます。位置が*1 か所*わかっている場合、これに相当するのが `method="nearest"` を付けた `.sel()` です。

In [ ]:
point = glorys["thetao"].sel(latitude=48.0, longitude=-5.0, method="nearest")
point.sel(depth=slice(0, 100)).plot(x="time", yincrease=False, cmap="RdYlBu_r", figsize=(8, 4))
plt.title("Temperature profile at 48N, 5W (depth 0 to 100 m)");

<div class="alert alert-success">
<b>ポイント</b>
<ul>
<li><code>.sel()</code> はラベルで、<code>.isel()</code> は位置で選択します。<code>method="nearest"</code> は最も近い格子点を選びます。</li>
<li><code>.plot()</code> は静的な図を、<code>.hvplot()</code> はインタラクティブな図を作ります。</li>
</ul>
</div>

## 3. Dask：遅延計算と並列計算

```{note}
ワークショップでは任意です。時間が足りない場合はこの節を飛ばします。この節だけで実行できるので、あとで取り組んでみてください。
```


1 週間の平均海面水温を計算してみましょう。式を書いただけでは、何も計算され**ません**。xarray と Dask は*タスクグラフ*を組み立てるだけです。

In [ ]:
weekly_mean = sst.mean("time")
weekly_mean  # まだ遅延評価のまま：Dask 配列の表示を見てみよう

`.compute()` は Dask クラスター上でグラフを実行し、結果を（NumPy 配列として）メモリに返します。実行中はダッシュボードを見てください。小さな四角形の一つひとつが、ワーカーが読み込んで平均しているチャンクです。

In [ ]:
%%time
weekly_mean = weekly_mean.compute()
weekly_mean

In [ ]:
weekly_mean.plot(cmap="RdYlBu_r", figsize=(8, 6))
plt.title("Weekly mean sea surface temperature");

ほかの計算もすべて同じように動きます。`.std("time")`、`.max("depth")`、`(a - b)`、`.where(condition)` などです。数式を書けば、チャンクの扱いと並列化は Dask が引き受けます。`pangeo-fish` はまさにこの仕組みを使って、すべての格子点・すべての深さ・すべての時刻について、タグの水温とモデルの水温の差を計算しています。

*演習（時間があれば）*：3 次元の場全体の週平均 `glorys["thetao"].mean("time")` を計算し、ダッシュボードを見てみましょう。はるかに多くのチャンクが使われます。

<div class="alert alert-success">
<b>ポイント</b>
<ul>
<li>Dask を使った配列の操作は遅延評価です。<code>.compute()</code>（または描画）によって計算が実行されます。</li>
<li>Dask のダッシュボードで、何が起きているかを確認できます。</li>
</ul>
</div>

## 4. Zarr：データセットを保存する（そして再び開く）

```{note}
ワークショップでは任意です。時間が足りない場合はこの節を飛ばします。この節だけで実行できるので、あとで取り組んでみてください。
```


Zarr は、各チャンクを別々の圧縮ファイルとして保存し、メタデータを小さな JSON ファイルに保存します。Copernicus Marine がクラウドで使っている形式であり、`pangeo-fish` がすべての途中結果（`diff.zarr`、`emission.zarr` など）を書き出す形式でもあります。そのため、ワークフローの各ステップを途中から再開できます。

1 週間分の海面水温のサブセットを、ローカルに保存してみましょう。データが（やはり並列で）ダウンロードされ、ディスクに書き込まれます。

In [ ]:
%%time
# drop_encoding()：リモートのストアの圧縮設定を破棄し、Zarr に選ばせる
sst.to_dataset(name="thetao").drop_encoding().to_zarr("glorys_sst.zarr", mode="w")

Zarr ストアはディレクトリです。中身を見てみましょう。

In [ ]:
store = pathlib.Path("glorys_sst.zarr")
files = sorted(p for p in store.rglob("*") if p.is_file())
print(len(files), "files, total", round(sum(p.stat().st_size for p in files) / 1e6, 1), "MB")
for p in files[:12]:
    print(p.relative_to(store))

`xr.open_zarr()` で再び開きます。リモートのデータと同じように、データセットは再び遅延評価（Dask を使用）になります。Zarr を使うと、ローカルのデータもクラウドのデータも同じように扱えます。

In [ ]:
sst_local = xr.open_zarr("glorys_sst.zarr")
sst_local

<div class="alert alert-success">
<b>ポイント</b>
<ul>
<li><code>.to_zarr()</code> で書き込み、<code>xr.open_zarr()</code> で読み込みます。どちらもチャンクを意識して並列に動きます。</li>
<li>リモート（クラウド）の Zarr もローカルの Zarr も、同じように扱えます。</li>
</ul>
</div>

## 5. HEALPix：魚の追跡のための等面積格子

GLORYS は規則的な緯度経度格子の上にあります。このような格子には、移動モデルにとって不都合な点があります。極に近づくほど格子が小さくなるのです（1/12° の格子は、赤道では約 9 km × 9 km ですが、北緯 60 度では 9 km × 4.5 km です）。`pangeo-fish` は魚が 1 日に泳げる距離を制限します。これは、**すべての格子が同じ面積**で、**隣り合う格子が同じ距離**にあるほうがずっと簡単です。

[HEALPix](https://healpix.sourceforge.io/)（Hierarchical Equal Area isoLatitude Pixelisation）は、球面を 12 個の基本格子に分け、*解像度レベル*が 1 つ上がるごとに各格子を 4 つに分割します。

| レベル | 格子の数 | 格子の大きさ（目安） |
|---|---|---|
| 0 | 12 | 6 400 km |
| 5 | 12 288 | 200 km |
| 8 | 786 432 | 25 km |
| 10 | 12 582 912 | 6 km |
| 12 | 201 326 592 | 1.6 km |

HEALPix は*離散全球格子システム*（DGGS）の一つです。ほかに H3（六角形）や rHEALPix などがあります。Python パッケージ [`xdggs`](https://github.com/xarray-contrib/xdggs) は、xarray に DGGS の機能を追加します。データセットは `latitude`/`longitude` の代わりに `cell_ids` という座標を持ち、`xdggs` は格子の中心や多角形、隣接する格子を計算したり、描画したりできます。

### GLORYS を HEALPix に変換する

ここでは [`healpix-resample`](https://github.com/GRID4EARTH/healpix-resample) を使います。[GRID4EARTH](https://grid4earth.eu/) プロジェクトで開発された小さなパッケージで（`pangeo-fish` でも使われています）、任意の経度・緯度の点の値を HEALPix 格子へ内挿します。手順は 3 つです。

1. GLORYS の格子点の経度と緯度をリサンプラーに渡します（内挿の重みは 1 回だけ計算されます）。
2. 値に対して `resample()` を呼びます（1 行が 1 時刻）。
3. 結果を `cell_ids` 座標を持つ `xarray.Dataset` に戻し、どの格子なのかを `xdggs` に伝えます。

下の小さな関数はこの 3 つの手順をまとめたもので、変換を 1 行で呼び出せるようにします。`BilinearResampler` は最も簡単な内挿法です。このパッケージには、最近傍、バイキュービック、保存型（conservative）などのリサンプラーもあり、同じ使い方ができます。`pangeo-fish`（次のノートブックの `regrid_dataset_hpresample`）は、海岸線をきちんと扱うために、これらのいくつかを組み合わせて使います。

In [ ]:
def to_healpix(ds, level, var="thetao", ellipsoid="WGS84"):
    # (time, latitude, longitude) の変数を HEALPix 格子へ内挿する
    # 1. 格子点を、経度と緯度の 1 次元配列にする
    lon2d, lat2d = np.meshgrid(ds["longitude"].values, ds["latitude"].values)
    resampler = BilinearResampler(
        lon_deg=lon2d.ravel().astype("float64"),
        lat_deg=lat2d.ravel().astype("float64"),
        level=level,
        verbose=False,
    )
    # 2. 内挿する：行 = 時刻、列 = 格子点
    values = ds[var].values.reshape(ds.sizes["time"], -1)
    result = resampler.resample(values)
    # 3. latitude/longitude の代わりに cell_ids を持つ xarray に戻す
    grid_info = {"grid_name": "healpix", "level": level, "indexing_scheme": "nested", "ellipsoid": ellipsoid}
    return xr.Dataset(
        {var: (("time", "cell_ids"), result.cell_data, ds[var].attrs)},
        coords={"time": ds["time"], "cell_ids": ("cell_ids", result.cell_ids.astype("int64"), grid_info)},
    )


refinement_level = 8  # 格子は約 25 km。pangeo-fish では通常 10〜12 を使う
sst_healpix = to_healpix(sst_local.compute(), refinement_level)
sst_healpix

データセットの空間次元は、`latitude` と `longitude` の代わりに `cell_ids` の 1 つだけになりました。`cell_ids` の属性が格子を記述しています。`xdggs.decode()` はそれを読み取り、`.dggs` アクセサーを使えるようにします。

In [ ]:
sst_healpix = xdggs.decode(sst_healpix)
sst_healpix.dggs.grid_info

In [ ]:
# 格子 ID から xdggs が計算した格子の中心
sst_healpix.dggs.cell_centers()

### HEALPix 格子の上に描く

`.dggs.explore()` は、各格子を多角形としてインタラクティブな地図上に描きます（拡大すると、等面積の格子が見えます。格子が経線に沿って並んでいないことに注目してください）。

In [ ]:
sst_healpix["thetao"].isel(time=0).dggs.explore(cmap="RdYlBu_r", alpha=0.8)

HEALPix のデータセットを Zarr として保存します。これは、次のノートブックのステップ 3 で `pangeo-fish` が作るファイル（`diff-regridded.zarr`）とまったく同じ種類のものです。`.dggs.encode("xdggs")` は格子の情報を `cell_ids` の属性に書き戻すので、読み込んだあとで再びデコードできます。

In [ ]:
sst_healpix.dggs.encode("xdggs").drop_encoding().to_zarr("glorys_sst_healpix.zarr", mode="w")
xr.open_zarr("glorys_sst_healpix.zarr").pipe(xdggs.decode)

<div class="alert alert-success">
<b>ポイント</b>
<ul>
<li>HEALPix の格子はすべて同じ面積を持ちます。これは移動モデルに必要な性質です。</li>
<li><code>healpix-resample</code>（GRID4EARTH）を使うと、任意の経度・緯度のデータを 3 行で HEALPix に内挿できます。</li>
<li><code>xdggs</code> では、HEALPix のデータセットは格子を記述する属性付きの <code>cell_ids</code> 座標を持ちます。<code>xdggs.decode()</code> で <code>.dggs</code> アクセサー（格子の中心、多角形、<code>explore()</code>）が使えるようになります。</li>
<li><code>pangeo-fish</code> は、モデルとタグの比較結果を HEALPix に再格子化し、Zarr として保存して、その格子の上で位置推定を行います。</li>
</ul>
</div>

## 6. 後片付け

次のノートブックに進む前に、Dask クラスターを閉じます（各ノートブックはそれぞれ自分のクラスターを起動します）。

In [ ]:
client.close()

## さらに学ぶために

- [Xarray チュートリアル](https://tutorial.xarray.dev/)と [Pangeo ギャラリー](https://gallery.pangeo.io/)。
- [Dask チュートリアル](https://tutorial.dask.org/)。
- [Copernicus Marine toolbox のドキュメント](https://toolbox-docs.marine.copernicus.eu/)。
- [xdggs](https://xdggs.readthedocs.io/) と [DGGS コミュニティ](https://github.com/xarray-contrib/xdggs)。
- [GRID4EARTH](https://grid4earth.eu/) のパッケージ：[healpix-geo](https://github.com/GRID4EARTH/healpix-geo)（格子 ID、幾何計算）、[healpix-resample](https://github.com/GRID4EARTH/healpix-resample)（上で使用）、[healpix-convert](https://github.com/GRID4EARTH/healpix-convert)（ERA5 や CAMS などの Zarr データセット全体を、Dask を使って HEALPix DGGS の Zarr 形式に一括変換）、[healpix-plot](https://github.com/GRID4EARTH/healpix-plot)（静的な地図）。
- HEALPix の論文：Górski et al. (2005), *HEALPix: A Framework for High-Resolution Discretization and Fast Analysis of Data Distributed on the Sphere*, ApJ 622, 759, [doi:10.1086/427976](https://doi.org/10.1086/427976)。